# Pipeline de ML — Predição de Falhas em Nós HPC

1. Carregamento (apenas colunas necessárias, GPU nodes, excluindo CANCELLED)
2. Rotulação (snapshots dentro das 2h antes do `end_date` de jobs de falha → label=1)
3. Split temporal 80/20 por `end_date` do job
4. Descarte de jobs com <40 snapshots
5. Sliding window (size=40) por `slurm_id`, agregação min/max/mean/std; label da janela por **maioria**
6. Treino: Random Forest + Histogram-Based Gradient Boosting (`class_weight=balanced`)
7. Avaliação: PR-AUC (primária), ROC-AUC, classification report, breakdown por state

In [1]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    classification_report,
)

DATASET_PATH = '../dataset/prom_slurm_joined/prom_slurm_joined.parquet'

FEATURES = [
    'node_memory_Active_bytes',
    'node_disk_written_bytes_total_sum',
    'node_netstat_Tcp_InErrs',
    'node_forks_total',
    'nvidia_gpu_temperature_celsius_mean',
    'nvidia_gpu_power_usage_milliwatts_mean',
    'nvidia_gpu_memory_used_bytes_sum',
    'nvidia_gpu_fanspeed_percent_mean',
]

SCAFFOLD = ['slurm_id', 'state', 'end_date', 'timestamp']
FAILURE_STATES = {'FAILED', 'TIMEOUT', 'OUT_OF_MEMORY', 'NODE_FAIL'}

WINDOW_SIZE = 40              # ~10 min de snapshots (15s cada)
PRE_FAILURE_HOURS = 2         # horizonte de predição
TRAIN_FRACTION = 0.8          # split temporal 80/20

## 1. Carregamento

Lemos apenas as colunas necessárias, filtrando para nós GPU e excluindo jobs cancelados.

In [2]:
columns_to_load = FEATURES + SCAFFOLD

dataset = pq.ParquetDataset(
    DATASET_PATH,
    filters=[('gpu_node', '=', 1)],
)

df = dataset.read(columns=columns_to_load).to_pandas()
df = df[df['state'] != 'CANCELLED'].reset_index(drop=True)

df['timestamp'] = pd.to_datetime(df['timestamp'])
df['end_date'] = pd.to_datetime(df['end_date'])

print(f'Snapshots: {len(df):,}')
print(f'Jobs (slurm_id únicos): {df["slurm_id"].nunique():,}')
print('\nDistribuição por state:')
print(df['state'].value_counts())

Snapshots: 11,144,842
Jobs (slurm_id únicos): 23,618

Distribuição por state:
state
COMPLETED        6769458
TIMEOUT          3643280
FAILED            631137
OUT_OF_MEMORY      73016
NODE_FAIL          27951
Name: count, dtype: int64


## 2. Rotulação

Para cada snapshot, `label=1` se o job termina em falha **e** o snapshot ocorre dentro das 2h antes do `end_date`.

In [3]:
is_failure_job = df['state'].isin(FAILURE_STATES)
time_to_end = df['end_date'] - df['timestamp']
in_pre_failure = (time_to_end <= pd.Timedelta(hours=PRE_FAILURE_HOURS)) & \
                 (time_to_end >= pd.Timedelta(0))
df['label'] = (is_failure_job & in_pre_failure).astype(np.int8)

print('Distribuição de labels (snapshots):')
print(df['label'].value_counts())
print(f'\nProporção de positivos: {df["label"].mean():.4f}')

Distribuição de labels (snapshots):
label
0    10718868
1      425974
Name: count, dtype: int64

Proporção de positivos: 0.0382


## 3. Split temporal 80/20 por `end_date`

Corte por `end_date` do job: jobs que **terminaram** antes do percentil 80 vão pro treino; demais vão pro teste. Isso garante que todo o ciclo de vida do job está de um único lado da fronteira temporal — sem vazamento.

In [4]:
job_end = df.groupby('slurm_id')['end_date'].first().sort_values()
n_train_jobs = int(len(job_end) * TRAIN_FRACTION)

train_jobs = set(job_end.iloc[:n_train_jobs].index)
test_jobs = set(job_end.iloc[n_train_jobs:].index)

df_train = df[df['slurm_id'].isin(train_jobs)].copy()
df_test = df[df['slurm_id'].isin(test_jobs)].copy()

split_time = job_end.iloc[n_train_jobs]
print(f'Corte temporal (end_date): {split_time}')
print(f'Jobs treino: {len(train_jobs):,}  |  Jobs teste: {len(test_jobs):,}')
print(f'Snapshots treino: {len(df_train):,}  |  Snapshots teste: {len(df_test):,}')

# Verifica ausência de vazamento
train_end_max = df_train['timestamp'].max()
test_start_min = df_test['timestamp'].min()
print(f'\nÚltimo timestamp em treino:  {train_end_max}')
print(f'Primeiro timestamp em teste: {test_start_min}')
print(f'Sobreposição temporal: {train_end_max > test_start_min}')

Corte temporal (end_date): 2022-10-19 11:50:16
Jobs treino: 18,894  |  Jobs teste: 4,724
Snapshots treino: 9,613,106  |  Snapshots teste: 1,531,736

Último timestamp em treino:  2022-10-19 11:47:30
Primeiro timestamp em teste: 2022-10-15 15:51:30
Sobreposição temporal: True


## 4 + 5. Janelamento e agregação

Para cada job: ordena por timestamp; descarta se tiver <40 snapshots; aplica sliding window de tamanho 40; agrega min/max/mean/std por feature (8 × 4 = 32 colunas). **Label da janela = maioria dos labels dos 40 snapshots** (≥20 positivos → janela positiva).

In [5]:
def build_windows(df_split, features, window_size):
    """Sliding window por slurm_id; agrega min/max/mean/std; label por maioria."""
    X_parts, y_parts = [], []
    df_split = df_split.sort_values(['slurm_id', 'timestamp'])
    n_jobs_total = df_split['slurm_id'].nunique()
    n_jobs_kept = 0
    majority_threshold = window_size // 2  # >=20 positivos em 40

    for _, job_df in df_split.groupby('slurm_id', sort=False):
        if len(job_df) < window_size:
            continue
        n_jobs_kept += 1

        arr = job_df[features].to_numpy(dtype=np.float32)
        labels = job_df['label'].to_numpy()

        windows = np.lib.stride_tricks.sliding_window_view(
            arr, window_shape=window_size, axis=0
        )
        agg = np.concatenate([
            windows.min(axis=-1),
            windows.max(axis=-1),
            windows.mean(axis=-1),
            windows.std(axis=-1),
        ], axis=1)

        # Label da janela = maioria (>= window_size//2 positivos)
        label_windows = np.lib.stride_tricks.sliding_window_view(labels, window_size)
        window_labels = (label_windows.sum(axis=1) >= majority_threshold).astype(np.int8)

        X_parts.append(agg)
        y_parts.append(window_labels)

    X = np.vstack(X_parts)
    y = np.concatenate(y_parts)
    print(f'  Jobs mantidos: {n_jobs_kept:,} / {n_jobs_total:,} '
          f'({n_jobs_kept / n_jobs_total:.1%})')
    print(f'  Janelas geradas: {len(X):,}  |  Positivas: {y.sum():,} ({y.mean():.4f})')
    return X, y


print('Treino:')
X_train, y_train = build_windows(df_train, FEATURES, WINDOW_SIZE)
print('\nTeste:')
X_test, y_test = build_windows(df_test, FEATURES, WINDOW_SIZE)

print(f'\nX_train: {X_train.shape}  |  X_test: {X_test.shape}')

Treino:
  Jobs mantidos: 10,320 / 18,894 (54.6%)
  Janelas geradas: 9,125,945  |  Positivas: 291,936 (0.0320)

Teste:
  Jobs mantidos: 2,694 / 4,724 (57.0%)
  Janelas geradas: 1,411,809  |  Positivas: 56,831 (0.0403)

X_train: (9125945, 32)  |  X_test: (1411809, 32)


## 6. Treino

In [16]:
rf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42,
)

hgb = HistGradientBoostingClassifier(
    max_iter=200,
    class_weight='balanced',
    random_state=42,
)

print('Treinando Random Forest...')
rf.fit(X_train, y_train)
print('Treinando Histogram Gradient Boosting...')
hgb.fit(X_train, y_train)
print('OK.')

Treinando Random Forest...
Treinando Histogram Gradient Boosting...
OK.


## 7. Avaliação

In [17]:
def evaluate(name, model, X, y):
    proba = model.predict_proba(X)[:, 1]
    pred = (proba >= 0.5).astype(int)

    pr_auc = average_precision_score(y, proba)
    roc_auc = roc_auc_score(y, proba)

    print(f'\n=== {name} ===')
    print(f'PR-AUC : {pr_auc:.4f}')
    print(f'ROC-AUC: {roc_auc:.4f}')
    print(classification_report(y, pred, digits=4,
                                target_names=['healthy (0)', 'pre-failure (1)']))
    return proba


proba_rf = evaluate('Random Forest', rf, X_test, y_test)
proba_hgb = evaluate('Histogram Gradient Boosting', hgb, X_test, y_test)


=== Random Forest ===
PR-AUC : 0.0584
ROC-AUC: 0.6187
                 precision    recall  f1-score   support

    healthy (0)     0.9597    0.9869    0.9731   1354978
pre-failure (1)     0.0345    0.0112    0.0169     56831

       accuracy                         0.9476   1411809
      macro avg     0.4971    0.4990    0.4950   1411809
   weighted avg     0.9224    0.9476    0.9346   1411809


=== Histogram Gradient Boosting ===
PR-AUC : 0.0652
ROC-AUC: 0.6106
                 precision    recall  f1-score   support

    healthy (0)     0.9706    0.6540    0.7814   1354978
pre-failure (1)     0.0601    0.5271    0.1078     56831

       accuracy                         0.6489   1411809
      macro avg     0.5153    0.5906    0.4446   1411809
   weighted avg     0.9339    0.6489    0.7543   1411809



## 8. Breakdown por state

Performance do HGB separada por desfecho final do job.

In [18]:
states_per_window = []
for _, job_df in df_test.sort_values(['slurm_id', 'timestamp']).groupby('slurm_id', sort=False):
    if len(job_df) < WINDOW_SIZE:
        continue
    states_per_window.extend([job_df['state'].iloc[0]] * (len(job_df) - WINDOW_SIZE + 1))
states_per_window = np.array(states_per_window)

print('=== HGB - quebra por state ===')
print(f"{'state':<18}{'n_windows':>12}{'n_pos':>10}{'recall@0.5':>14}{'mean_proba':>14}")
print('-' * 68)
for state in ['TIMEOUT', 'FAILED', 'OUT_OF_MEMORY', 'NODE_FAIL', 'COMPLETED']:
    mask = states_per_window == state
    if mask.sum() == 0:
        continue
    y_sub = y_test[mask]
    p_sub = proba_hgb[mask]
    n_pos = int(y_sub.sum())
    recall_str = f'{((p_sub >= 0.5) & (y_sub == 1)).sum() / n_pos:.4f}' if n_pos > 0 else 'n/a'
    print(f'{state:<18}{mask.sum():>12,}{n_pos:>10,}{recall_str:>14}{p_sub.mean():>14.4f}')

=== HGB - quebra por state ===
state                n_windows     n_pos    recall@0.5    mean_proba
--------------------------------------------------------------------
TIMEOUT                408,660    39,857        0.5487        0.4558
FAILED                  51,962    12,862        0.4603        0.4457
OUT_OF_MEMORY            1,606     1,592        0.6212        0.6231
NODE_FAIL               23,159     2,520        0.4679        0.4381
COMPLETED              926,422         0           n/a        0.3666


- Testar 1D-CNN também
- Ver hiperparâmetros de Skrzeckek
- Reproduzir Skrzeckek 
- Deixar apresentação inteligível

- ver dia, hora e profs da banca